<a href="https://colab.research.google.com/github/arturzxc228/student-perfomance-analysys/blob/main/Multi_class_Classification_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text classification with Naive Bayes Classifier



In [ ]:
!pip install pymorphy3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm, trange

import pymorphy3
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import Counter

In [ ]:
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## Загружаем и готовим набор данных

In [ ]:
!wget https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.1/lenta-ru-news.csv.bz2 -O lenta-ru-news.csv.bz2

--2025-09-30 09:13:25--  https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.1/lenta-ru-news.csv.bz2
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/87156914/619f9f00-1e96-11ea-946e-dac89df8aced?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-09-30T10%3A04%3A25Z&rscd=attachment%3B+filename%3Dlenta-ru-news.csv.bz2&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-09-30T09%3A03%3A37Z&ske=2025-09-30T10%3A04%3A25Z&sks=b&skv=2018-11-09&sig=6kaH7FurzUKsFpgzkSSmnfDUi0wuGlA2D1xB5G0eTxg%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc1OTIyMzkwNSwibmJmIjoxNzU5MjIzNjA1LCJwYXRoIjoicmVsZWFzZWFzc2V0

In [ ]:
news = pd.read_csv('lenta-ru-news.csv.bz2');

/tmp/ipython-input-308735011.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  news = pd.read_csv('lenta-ru-news.csv.bz2');


In [ ]:
news

,url,title,text,topic,tags,date
0,https://lenta.ru/news/1914/09/16/hungarnn/,1914. Русские войска вступили в пределы Венгрии,Бои у Сопоцкина и Друскеник закончились отступ...,Библиотека,Первая мировая,1914/09/16
1,https://lenta.ru/news/1914/09/16/lermontov/,1914. Празднование столетия М.Ю. Лермонтова от...,"Министерство народного просвещения, в виду про...",Библиотека,Первая мировая,1914/09/16
2,https://lenta.ru/news/1914/09/17/nesteroff/,1914. Das ist Nesteroff!,"Штабс-капитан П. Н. Нестеров на днях, увидев в...",Библиотека,Первая мировая,1914/09/17
3,https://lenta.ru/news/1914/09/17/bulldogn/,1914. Бульдог-гонец под Льежем,Фотограф-корреспондент Daily Mirror рассказыва...,Библиотека,Первая мировая,1914/09/17
4,https://lenta.ru/news/1914/09/18/zver/,1914. Под Люблином пойман швабский зверь,"Лица, приехавшие в Варшаву из Люблина, передаю...",Библиотека,Первая мировая,1914/09/18
...,...,...,...,...,...,...
800970,https://lenta.ru/news/2019/12/14/shnur/,Шнуров раскритиковал Гагарину на «Голосе»,Певец Сергей Шнуров раскритиковал свою коллегу...,NaN,ТВ и радио,2019/12/14
800971,https://lenta.ru/news/2019/12/14/dolg/,В России предложили изменить правила взыскания...,Министерство юстиции России предложило изменит...,NaN,Все,2019/12/14
800972,https://lenta.ru/news/2019/12/14/dark_euro/,В России назвали «черную дату» для Европы,Испытание США ранее запрещенной Договором о ли...,NaN,Политика,2019/12/14
800973,https://lenta.ru/news/2019/12/14/meteo/,Россиянам пообещали аномально теплую погоду,В ближайшие дни в европейской части России пог...,NaN,Общество,2019/12/14


In [ ]:
df_sorted = news.sort_values(by="date", ascending=False)

In [ ]:
df_sorted

,url,title,text,topic,tags,date
800974,https://lenta.ru/news/2019/12/14/olimp/,В конкурсе прогнозов на АПЛ разыграют 100 тыся...,Ведущие футбольные чемпионаты ушли на зимние к...,NaN,Английский футбол,2019/12/14
800940,https://lenta.ru/news/2019/12/14/opr/,Приставы прокомментировали данные о 25 миллион...,Федеральная служба судебных приставов (ФССП) п...,NaN,Общество,2019/12/14
800949,https://lenta.ru/news/2019/12/14/uroki/,В России назвали число запретивших смартфоны н...,В каждой четвертой российской школе запретили ...,NaN,Общество,2019/12/14
800948,https://lenta.ru/news/2019/12/14/violence/,Законопроект о борьбе с домашним насилием изменят,Законопроект о борьбе с домашним насилием одно...,NaN,Политика,2019/12/14
800947,https://lenta.ru/news/2019/12/14/strelba/,Двое россиян устроили стрельбу в двух барах за...,В башкирском Белебее двое россиян за одну ночь...,NaN,Криминал,2019/12/14
...,...,...,...,...,...,...
4,https://lenta.ru/news/1914/09/18/zver/,1914. Под Люблином пойман швабский зверь,"Лица, приехавшие в Варшаву из Люблина, передаю...",Библиотека,Первая мировая,1914/09/18
2,https://lenta.ru/news/1914/09/17/nesteroff/,1914. Das ist Nesteroff!,"Штабс-капитан П. Н. Нестеров на днях, увидев в...",Библиотека,Первая мировая,1914/09/17
3,https://lenta.ru/news/1914/09/17/bulldogn/,1914. Бульдог-гонец под Льежем,Фотограф-корреспондент Daily Mirror рассказыва...,Библиотека,Первая мировая,1914/09/17
1,https://lenta.ru/news/1914/09/16/lermontov/,1914. Празднование столетия М.Ю. Лермонтова от...,"Министерство народного просвещения, в виду про...",Библиотека,Первая мировая,1914/09/16


In [ ]:
df_filtered = news[(news["date"] >= "2000/01/01") & (news["date"] <= "2019/12/31")]

In [ ]:
df_filtered

NameError: name 'df_filtered' is not defined

In [ ]:
print(news['tags'].unique(), len(news['tags'].unique()))

['Первая мировая' 'Все' nan 'Прибалтика' 'Кино' 'Преступность' 'Общество'
 'Происшествия' 'Искусство' 'Бизнес' 'Техника' 'ТВ и радио' 'Политика'
 'Пресса' 'Музыка' 'Люди' 'Звери' 'Игры' 'Госэкономика' 'Гаджеты' 'Наука'
 'Еда' 'Рынки' 'Деньги' 'Летние виды' 'Интернет' 'Театр' 'Конфликты'
 'Реклама' 'Космос' 'Бокс и ММА' 'Футбол' 'Книги' 'Зимние виды'
 'Достижения' 'Coцсети' 'Вещи' 'События' 'Средняя Азия' 'Украина'
 'Закавказье' 'Белоруссия' 'Молдавия' 'Софт' 'Квартира' 'Город' 'Дача'
 'Офис' 'Оружие' 'Мнения' 'Москва' 'Регионы' 'Полиция и спецслужбы'
 'Криминал' 'Следствие и суд' 'Движение' 'Производители' 'Мировой бизнес'
 'Финансы компаний' 'Деловой климат' 'Мир' 'Россия' 'Часы' 'Явления'
 'Стиль' 'Инструменты' 'Вооружение' 'Вкусы' 'Страноведение'
 'Госрегулирование' 'История' 'Внешний вид' 'Автобизнес' 'Аналитика рынка'
 'Туризм' 'Выборы' 'Экология' 'Мемы' 'Мировой опыт' 'Инновации' 'Хоккей'
 'Вирусные ролики' 'Фотография' 'Авто' 'Наследие' 'Преступная Россия'
 'Жизнь' 'Киберпреступ

In [ ]:
print(news['topic'].unique(), len(news['topic'].unique()))

['Библиотека' 'Россия' 'Мир' 'Экономика' 'Интернет и СМИ' 'Спорт'
 'Культура' 'Из жизни' 'Силовые структуры' 'Наука и техника' 'Бывший СССР'
 nan 'Дом' 'Сочи' 'ЧМ-2014' 'Путешествия' 'Ценности' 'Легпром' 'Бизнес'
 'МедНовости' 'Оружие' '69-я параллель' 'Культпросвет ' 'Крым'] 24


In [ ]:
topics = ['Путешествия', 'Ценности', 'Мир', 'Наука и техника', 'Экономика']
news_in_cat_count = 1000

In [ ]:
df_res = pd.DataFrame()

for topic in tqdm(topics):
    df_topic = news[news['topic'] == topic][:news_in_cat_count]
    df_res = pd.concat([df_res, df_topic], ignore_index=True)

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
df_res.shape

(5000, 6)

In [ ]:
def preprocess(text, stop_words, punctuation_marks, morph):
    tokens = word_tokenize(text.lower())
    preprocessed_text = []
    for token in tokens:
        if token not in punctuation_marks:
            lemma = morph.parse(token)[0].normal_form
            if lemma not in stop_words:
                preprocessed_text.append(lemma)
    return preprocessed_text

In [ ]:
punctuation_marks = ['!', ',', '(', ')', ':', '-', '?', '.', '..', '...', '«', '»', ';', '–', '--']
stop_words = stopwords.words("russian")
morph = pymorphy3.MorphAnalyzer()

In [ ]:
df_res['Preprocessed_texts'] = df_res.apply(lambda row: preprocess(row['text'], punctuation_marks, stop_words, morph), axis=1)

In [ ]:
df_res.head()

,url,title,text,topic,tags,date,Preprocessed_texts
0,https://lenta.ru/news/2014/06/26/annoyingphotos/,Составлен рейтинг самых раздражающих отпускных...,"Названы самые раздражающие фотографии, которые...",Путешествия,Мнения,2014/06/26,"[назвать, самый, раздражающий, фотография, кот..."
1,https://lenta.ru/news/2015/01/09/flight/,В Европе появится 10-минутный авиарейс,Австрийская авиакомпания FlyNiki открывает 10-...,Путешествия,Мир,2015/01/09,"[австрийский, авиакомпания, flyniki, открывать..."
2,https://lenta.ru/news/2015/01/28/tsipras/,Греческий премьер призвал избавиться от отелей...,Премьер-министр Греции Алексис Ципрас призвал ...,Путешествия,Мир,2015/01/28,"[премьер-министр, греция, алексис, ципраса, пр..."
3,https://lenta.ru/news/2015/02/03/standfly/,Китайская авиакомпания собралась ввести в само...,Китайский лоукостер Spring Airlines собрался п...,Путешествия,Мир,2015/02/03,"[китайский, лоукостер, spring, airlines, собра..."
4,https://lenta.ru/news/2015/02/12/tripforvalent...,Россияне отправятся отмечать День святого Вале...,Большинство российских путешественников отметя...,Путешествия,Россия,2015/02/12,"[большинство, российский, путешественник, отме..."


In [ ]:
X = df_res['Preprocessed_texts']
y = df_res['topic']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state = 42)

In [ ]:
my_tags = df_res['topic'].unique()
my_tags

array(['Путешествия', 'Ценности', 'Мир', 'Наука и техника', 'Экономика'],
      dtype=object)

Naive Bayes Classifier

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
nb = Pipeline([('vect', CountVectorizer()),
               ('tfidf', TfidfTransformer()),
               ('clf', MultinomialNB()),
              ])

In [ ]:
X_train = pd.Series(X_train).astype(str)
X_test= pd.Series(X_test).astype(str)

In [ ]:
%%time
nb.fit(X_train, y_train)

CPU times: user 703 ms, sys: 15 ms, total: 718 ms
Wall time: 747 ms


Pipeline(steps=[('vect', CountVectorizer()), ('tfidf', TfidfTransformer()),
                ('clf', MultinomialNB())])

In [ ]:
%%time
from sklearn.metrics import classification_report
y_pred = nb.predict(X_test)

CPU times: user 343 ms, sys: 0 ns, total: 343 ms
Wall time: 352 ms


In [ ]:
from sklearn.metrics import accuracy_score

print('accuracy %s' % accuracy_score(y_pred, y_test))
print(classification_report(y_test, y_pred, target_names=my_tags))

accuracy 0.9166666666666666
                 precision    recall  f1-score   support

    Путешествия       0.88      0.82      0.85       284
       Ценности       0.92      0.88      0.90       301
            Мир       0.91      0.98      0.94       316
Наука и техника       0.98      0.93      0.95       309
      Экономика       0.89      0.97      0.93       290

       accuracy                           0.92      1500
      macro avg       0.92      0.92      0.91      1500
   weighted avg       0.92      0.92      0.92      1500



Проверка на примерах новостей с сайта

In [ ]:
econ_text = '''
Доллар США считается символом надежности, безопасности и экономического процветания.
Он занимает неоспоримое доминирующее положение в международной финансовой системе с середины XX века и производит
впечатление непобедимого титана. Однако эра господства доллара как основной мировой резервной валюты медленно
подходит к концу. Крупнейшие банки предрекают ему резкий спад уже в следующем году, а известный экономист
Стивен Роуч уверен, что американская валюта может обесцениться на треть.
Причинами обвала станут сокращение сбережений населения, рост государственного долга США и усиление Китая.
Закат долларового диктата — в материале «Ленты.ру».
Непомерные привилегии

Успех американской экономики в XX веке был во многом обусловлен доминирующей ролью доллара. В свою очередь достижение этой роли стало результатом политического и военного превосходства, которое США приобрели после Первой мировой войны. До сих пор позиция доллара в мире финансов представляет собой главную основу процветания США. Однако в 2002 году наметилась долгосрочная тенденция к ослаблению американской валюты, которая наблюдается и по сей день.

Действительно, последнее время доллар чувствует себя не очень хорошо. В частности, июль оказался очень сложным месяцем для американской валюты, которая обновила многомесячные, а в некоторых случаях и многолетние минимумы. Всего за месяц доллар подешевел на шесть процентов против фунта, на пять — по отношению к евро, на четыре — против швейцарского франка и австралийского доллара. Вдобавок валютные аналитики вполне допускают, что июль может оказаться для доллара худшим месяцем с точки зрения месячной динамики за последние десять лет.

'''

In [ ]:
econ_preprocessed_text = preprocess(econ_text, stop_words, punctuation_marks, morph)

In [ ]:
econ_preprocessed_text

['доллар',
 'сша',
 'считаться',
 'символ',
 'надёжность',
 'безопасность',
 'экономический',
 'процветание',
 'занимать',
 'неоспоримый',
 'доминировать',
 'положение',
 'международный',
 'финансовый',
 'система',
 'середина',
 'xx',
 'век',
 'производить',
 'впечатление',
 'непобедимый',
 'титан',
 'однако',
 'эра',
 'господство',
 'доллар',
 'основной',
 'мировой',
 'резервный',
 'валюта',
 'медленно',
 'подходить',
 'конец',
 'крупный',
 'банк',
 'предрекать',
 'резкий',
 'спад',
 'следующий',
 'год',
 'известный',
 'экономист',
 'стивен',
 'роучий',
 'уверенный',
 'американский',
 'валюта',
 'мочь',
 'обесцениться',
 'треть',
 'причина',
 'обвал',
 'стать',
 'сокращение',
 'сбережение',
 'население',
 'рост',
 'государственный',
 'долг',
 'сша',
 'усиление',
 'китай',
 'закат',
 'долларовый',
 'диктат',
 '—',
 'материал',
 'лента.ру',
 'непомерный',
 'привилегия',
 'успех',
 'американский',
 'экономика',
 'xx',
 'век',
 'многое',
 'обусловить',
 'доминировать',
 'роль',
 'доллар',

In [ ]:
ect_pred = nb.predict([econ_text])
ect_pred

array(['Экономика'], dtype='<U15')